# Feedback Prize — Baseline 2: Sliding Window + Class Weights + Official Metric

Builds on the data already loaded by `eda.ipynb` / `baseline1.ipynb`. Fixes three issues found in `baseline1`:

1. **Long essays were truncated at 512 tokens** — 22.9% of essays lost their tail (often the `Concluding Statement`). This version uses **sliding-window chunking** (`return_overflowing_tokens` + `stride`) so every essay is fully covered.
2. **`Rebuttal`/`Counterclaim` were undertrained** — only the `O` class was downweighted before. This version computes **inverse-frequency class weights** for all 15 labels.
3. **The evaluation metric was an approximation.** This version implements the **exact competition rule**: a match requires overlap ≥ 0.5 × len(gt) **and** ≥ 0.5 × len(pred).


## 0. 📁 Project Paths (data already downloaded via `eda.ipynb`)

In [3]:
import os
from pathlib import Path
import pandas as pd

# Project paths — data already exists in /content/data/raw from eda.ipynb
ROOT_DIR = Path("/content")
DATA_RAW_DIR = ROOT_DIR / "data" / "raw"
CSV_PATH = DATA_RAW_DIR / "train.csv"
TXT_DIR = DATA_RAW_DIR / "train"
TEST_DIR = DATA_RAW_DIR / "test"

if not CSV_PATH.exists() or not TXT_DIR.exists():
    raise FileNotFoundError(
        f"❌ Data not found in {DATA_RAW_DIR}\n"
        "Run notebooks/eda.ipynb first to download the dataset."
    )

df = pd.read_csv(CSV_PATH)
print(f"✅ Project root: {ROOT_DIR}")
print(f"✅ train.csv: {CSV_PATH}")
print(f"✅ train/ folder: {TXT_DIR}")
print(f"📊 Rows loaded: {len(df):,}")
print(f"📄 Unique essays: {df['id'].nunique():,}")


✅ Project root: /content
✅ train.csv: /content/data/raw/train.csv
✅ train/ folder: /content/data/raw/train
📊 Rows loaded: 144,293
📄 Unique essays: 15,594


## 1. ⚙️ Imports, Seed & GPU Check

In [4]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print(f"⚡ Device: {device}")
if torch.cuda.is_available():
    print(f"   🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   💾 VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB")


⚡ Device: cuda
   🚀 GPU: Tesla T4
   💾 VRAM: 15.64 GB


## 2. 🏷️ Config & Label Scheme (BIO, 15 labels)

In [5]:
CFG = {
    "model_name": "microsoft/deberta-v3-small",
    "max_len": 512,          # hard ceiling: DeBERTa-v3 position embeddings only go to 512
    "doc_stride": 128,       # overlap between chunks so boundary tokens keep context
    "batch_size": 8,
    "eval_batch_size": 16,
    "eval_essay_chunk": 16,  # how many essays' chunks to build/predict at once
    "grad_accum_steps": 2,
    "epochs": 3,
    "learning_rate": 2e-5,
    "use_class_weights": True,
    "output_dir": "outputs/",
}
os.makedirs(CFG["output_dir"], exist_ok=True)

DISCOURSE_TYPES = [
    "Lead", "Position", "Claim", "Counterclaim",
    "Rebuttal", "Evidence", "Concluding Statement"
]

LABELS = ["O"]
for dt in DISCOURSE_TYPES:
    LABELS.append(f"B-{dt}")
    LABELS.append(f"I-{dt}")

LABEL2ID = {lbl: i for i, lbl in enumerate(LABELS)}
ID2LABEL = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
print(f"🏷️ Total BIO labels: {NUM_LABELS}")


🏷️ Total BIO labels: 15


## 3. 🔤 Data Utilities

Character-level helpers: mapping characters to whitespace-split word indices, and building a per-character label array directly from `discourse_start`/`discourse_end`.

In [6]:
def read_essay_text(essay_id, data_dir=TXT_DIR):
    path = os.path.join(str(data_dir), f"{essay_id}.txt")
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def create_char2word_map(text):
    """Maps each character index to a whitespace-split word index (-1 if whitespace)."""
    char2word = np.full(len(text), -1, dtype=int)
    word_idx = 0
    in_word = False
    for i, ch in enumerate(text):
        if not ch.isspace():
            char2word[i] = word_idx
            in_word = True
        else:
            if in_word:
                word_idx += 1
                in_word = False
    return char2word

def char_labels_for_essay(text, group_df):
    """Builds a per-character label-id array for one essay from its discourse rows."""
    char_labels = np.zeros(len(text), dtype=int)
    if group_df is not None:
        for _, row in group_df.iterrows():
            dt = row["discourse_type"]
            b_id = LABEL2ID[f"B-{dt}"]
            i_id = LABEL2ID[f"I-{dt}"]
            st, en = int(row["discourse_start"]), int(row["discourse_end"])
            if st < len(text) and en <= len(text) and st < en:
                char_labels[st:en] = i_id
                char_labels[st] = b_id
    return char_labels


## 4. 🪟 Sliding Window Chunking (Fix #1)

Instead of truncating each essay at 512 tokens, we tokenize with `return_overflowing_tokens=True` and a `stride` — long essays are split into multiple overlapping 512-token chunks, so every part of the essay (including the tail, where `Concluding Statement` usually lives) gets trained on.

In [7]:
def build_chunks_for_essay(eid, text, group_df, tokenizer, max_len=None, stride=None):
    max_len = max_len or CFG["max_len"]
    stride = stride or CFG["doc_stride"]
    char_labels = char_labels_for_essay(text, group_df)

    encoding = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    chunks = []
    for c in range(len(encoding["input_ids"])):
        offsets = encoding["offset_mapping"][c]
        labels = [-100] * len(offsets)
        for tok_idx, (st, en) in enumerate(offsets):
            if st == en:
                continue
            tok_chars = char_labels[st:en]
            b_matches = tok_chars[(tok_chars >= 1) & (tok_chars % 2 == 1)]
            if len(b_matches) > 0:
                labels[tok_idx] = int(b_matches[0])
            else:
                labels[tok_idx] = int(tok_chars[0])

        chunks.append({
            "essay_id": eid,
            "input_ids": encoding["input_ids"][c],
            "attention_mask": encoding["attention_mask"][c],
            "labels": labels,
        })
    return chunks

class FeedbackDataset(Dataset):
    """Precomputes all overlapping chunks for every essay up front."""

    def __init__(self, essay_ids, df, tokenizer, max_len=None, stride=None, data_dir=TXT_DIR):
        self.samples = []
        grouped = df.groupby("id") if df is not None else None
        for eid in tqdm(essay_ids, desc="Building chunks"):
            text = read_essay_text(eid, data_dir)
            group_df = grouped.get_group(eid) if (grouped is not None and eid in grouped.groups) else None
            self.samples.extend(build_chunks_for_essay(eid, text, group_df, tokenizer, max_len, stride))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            "input_ids": torch.tensor(s["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(s["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(s["labels"], dtype=torch.long),
        }


## 5. ⚖️ Class Weights for Imbalance (Fix #2)

EDA showed `Claim`/`Evidence` dominate (~35%/~32%) while `Rebuttal`/`Counterclaim` are rare (~3%/~4%). Instead of only downweighting `O` (as in baseline1), this computes inverse-frequency weights for **all 15 labels**, clipped to avoid destabilizing training.

In [8]:
def compute_class_weights(dataset, num_labels, clip=(0.2, 5.0)):
    counts = np.zeros(num_labels, dtype=np.float64)
    for s in dataset.samples:
        for l in s["labels"]:
            if l != -100:
                counts[l] += 1
    counts = np.clip(counts, 1, None)
    inv_freq = 1.0 / counts
    weights = inv_freq / inv_freq.sum() * num_labels
    weights = np.clip(weights, clip[0], clip[1])
    return torch.tensor(weights, dtype=torch.float)


## 6. 🧩 Span Extraction & Merging

Converts word-level BIO predictions into spans, gluing together same-type spans separated by a short run of `O` (noise cleanup).

In [9]:
def merge_spans(spans, max_gap=3):
    if not spans:
        return []
    merged = []
    curr_type, curr_str = spans[0]
    curr_indices = [int(x) for x in curr_str.split()]

    for next_type, next_str in spans[1:]:
        next_indices = [int(x) for x in next_str.split()]
        if next_type == curr_type and (next_indices[0] - curr_indices[-1] <= max_gap + 1):
            curr_indices = list(range(curr_indices[0], next_indices[-1] + 1))
        else:
            merged.append((curr_type, " ".join(map(str, curr_indices))))
            curr_type = next_type
            curr_indices = next_indices

    merged.append((curr_type, " ".join(map(str, curr_indices))))
    return merged

def word_labels_to_spans(word_preds):
    spans = []
    curr_type, curr_indices = None, []
    for idx, label in enumerate(word_preds):
        if label == "O":
            if curr_type is not None:
                spans.append((curr_type, " ".join(map(str, curr_indices))))
            curr_type, curr_indices = None, []
        elif label.startswith("B-"):
            if curr_type is not None:
                spans.append((curr_type, " ".join(map(str, curr_indices))))
            curr_type, curr_indices = label[2:], [idx]
        elif label.startswith("I-"):
            t = label[2:]
            if curr_type == t:
                curr_indices.append(idx)
            else:
                if curr_type is not None:
                    spans.append((curr_type, " ".join(map(str, curr_indices))))
                curr_type, curr_indices = t, [idx]
    if curr_type is not None:
        spans.append((curr_type, " ".join(map(str, curr_indices))))
    return merge_spans(spans, max_gap=3)


## 7. 🔮 Sliding-Window Batched Inference

For words covered by more than one overlapping chunk, keeps the prediction from whichever chunk the word sits most centrally in — the chunk with the most surrounding context for that word.

In [10]:
def predict_essay_batch(model, tokenizer, texts, eids, device,
                         max_len=None, stride=None, batch_size=None):
    max_len = max_len or CFG["max_len"]
    stride = stride or CFG["doc_stride"]
    batch_size = batch_size or CFG["eval_batch_size"]

    all_chunks = []
    char2word_maps = {}
    word_counts = {}

    for eid, text in zip(eids, texts):
        char2word_maps[eid] = create_char2word_map(text)
        word_counts[eid] = len(text.split())
        enc = tokenizer(
            text, truncation=True, max_length=max_len, stride=stride,
            return_overflowing_tokens=True, return_offsets_mapping=True,
            padding="max_length",
        )
        for c in range(len(enc["input_ids"])):
            all_chunks.append({
                "essay_id": eid,
                "input_ids": enc["input_ids"][c],
                "attention_mask": enc["attention_mask"][c],
                "offsets": enc["offset_mapping"][c],
            })

    word_label = {eid: ["O"] * word_counts[eid] for eid in eids}
    word_margin = {eid: [-1] * word_counts[eid] for eid in eids}

    model.eval()
    with torch.no_grad():
        for start in range(0, len(all_chunks), batch_size):
            batch = all_chunks[start:start + batch_size]
            input_ids = torch.tensor([b["input_ids"] for b in batch], dtype=torch.long).to(device)
            attention_mask = torch.tensor([b["attention_mask"] for b in batch], dtype=torch.long).to(device)

            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()

            for b_idx, b in enumerate(batch):
                eid = b["essay_id"]
                offsets = b["offsets"]
                char2word = char2word_maps[eid]

                valid_positions = [i for i, (st, en) in enumerate(offsets) if st != en]
                if not valid_positions:
                    continue
                lo, hi = valid_positions[0], valid_positions[-1]

                for tok_idx in valid_positions:
                    st, en = offsets[tok_idx]
                    if st >= len(char2word):
                        continue
                    w_idx = -1
                    for c in range(st, en):
                        if c < len(char2word) and char2word[c] != -1:
                            w_idx = char2word[c]
                            break
                    if w_idx == -1 or w_idx >= word_counts[eid]:
                        continue

                    margin = min(tok_idx - lo, hi - tok_idx)
                    if margin > word_margin[eid][w_idx]:
                        word_margin[eid][w_idx] = margin
                        word_label[eid][w_idx] = ID2LABEL[preds[b_idx][tok_idx]]

    return word_label


## 8. 🎯 Official Metric & Macro-F1 Evaluation (Fix #3)

Implements the exact Kaggle competition rule: a prediction matches iff overlap ≥ 0.5 × len(gt) **and** overlap ≥ 0.5 × len(pred) — stricter than a simple IoU.

In [11]:
def official_match(gt_idx_str, pred_idx_str):
    gt_set = set(gt_idx_str.split())
    pred_set = set(pred_idx_str.split())
    if not gt_set or not pred_set:
        return False
    inter = len(gt_set & pred_set)
    return (inter / len(gt_set) >= 0.5) and (inter / len(pred_set) >= 0.5)

def evaluate_macro_f1(model, tokenizer, val_ids, df_val, device, data_dir=TXT_DIR,
                       essay_batch_size=None):
    essay_batch_size = essay_batch_size or CFG["eval_essay_chunk"]
    gt_grouped = df_val.groupby("id")
    tp = {dt: 0 for dt in DISCOURSE_TYPES}
    fp = {dt: 0 for dt in DISCOURSE_TYPES}
    fn = {dt: 0 for dt in DISCOURSE_TYPES}

    val_ids = list(val_ids)
    for start in tqdm(range(0, len(val_ids), essay_batch_size), desc="Evaluating Val F1"):
        batch_ids = val_ids[start:start + essay_batch_size]
        texts = [read_essay_text(eid, data_dir) for eid in batch_ids]
        word_labels = predict_essay_batch(model, tokenizer, texts, batch_ids, device)

        for eid in batch_ids:
            pred_spans = word_labels_to_spans(word_labels[eid])
            gt_spans = []
            if eid in gt_grouped.groups:
                for _, row in gt_grouped.get_group(eid).iterrows():
                    gt_spans.append((row["discourse_type"], str(row["predictionstring"]).strip()))

            matched_pred, matched_gt = set(), set()
            for gi, (g_type, g_str) in enumerate(gt_spans):
                for pi, (p_type, p_str) in enumerate(pred_spans):
                    if pi in matched_pred or p_type != g_type:
                        continue
                    if official_match(g_str, p_str):
                        tp[g_type] += 1
                        matched_gt.add(gi)
                        matched_pred.add(pi)
                        break

            for gi, (g_type, _) in enumerate(gt_spans):
                if gi not in matched_gt:
                    fn[g_type] += 1
            for pi, (p_type, _) in enumerate(pred_spans):
                if pi not in matched_pred:
                    fp[p_type] += 1

    f1s, rows = [], []
    for dt in DISCOURSE_TYPES:
        denom = 2 * tp[dt] + fp[dt] + fn[dt]
        f1 = (2 * tp[dt]) / denom if denom > 0 else 0.0
        precision = tp[dt] / (tp[dt] + fp[dt]) if (tp[dt] + fp[dt]) > 0 else 0.0
        recall = tp[dt] / (tp[dt] + fn[dt]) if (tp[dt] + fn[dt]) > 0 else 0.0
        f1s.append(f1)
        rows.append({"Discourse Type": dt, "Precision": round(precision, 4),
                     "Recall": round(recall, 4), "F1-Score": round(f1, 4),
                     "TP": tp[dt], "FP": fp[dt], "FN": fn[dt]})
    macro_f1 = float(np.mean(f1s))
    return macro_f1, pd.DataFrame(rows)


## 9. 🧠 Model Setup

In [12]:
model = AutoModelForTokenClassification.from_pretrained(
    CFG["model_name"],
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)

# Some transformers/torch combos load checkpoints in fp16 by default. GradScaler
# requires fp32 master weights, so force fp32 explicitly regardless of what
# from_pretrained picked, to avoid "Attempting to unscale FP16 gradients."
model = model.float()

tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"], add_prefix_space=True)
print(f"✅ Model ready: {CFG['model_name']} ({NUM_LABELS} BIO labels)")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  286MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

✅ Model ready: microsoft/deberta-v3-small (15 BIO labels)


## 10. 📦 Train/Validation Split & Chunked Datasets

In [13]:
all_essay_ids = df["id"].unique().tolist()
train_ids, val_ids = train_test_split(all_essay_ids, test_size=0.15, random_state=SEED)

train_ds = FeedbackDataset(train_ids, df, tokenizer, data_dir=TXT_DIR)
print(f"📊 Train chunks (post sliding-window): {len(train_ds):,} (from {len(train_ids):,} essays)")

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

val_df = df[df["id"].isin(val_ids)].copy()
print(f"📊 Val essays: {len(val_ids):,}")


Building chunks:   0%|          | 0/13254 [00:00<?, ?it/s]

📊 Train chunks (post sliding-window): 18,979 (from 13,254 essays)
📊 Val essays: 2,340


## 11. ⚖️ Compute Class Weights & Loss Function

In [14]:
class_weights = None
if CFG["use_class_weights"]:
    class_weights = compute_class_weights(train_ds, NUM_LABELS).to(device)
    print("Class weights:", class_weights.cpu().numpy().round(2))

criterion = nn.CrossEntropyLoss(ignore_index=-100, weight=class_weights)


Class weights: [0.2  2.5  0.2  1.49 0.2  0.44 0.2  3.49 0.2  4.61 0.2  0.47 0.2  1.49
 0.2 ]


## 12. 🚀 Training Loop

In [15]:
def train_epoch(model, loader, optimizer, scheduler, device, criterion, scaler=None):
    model.train()
    total_loss = 0
    valid_batches = 0

    for step, batch in enumerate(tqdm(loader, desc="Training")):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(device_type="cuda", enabled=USE_AMP):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits.view(-1, NUM_LABELS), labels.view(-1))

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad()
            continue

        loss = loss / CFG["grad_accum_steps"]
        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % CFG["grad_accum_steps"] == 0 or (step + 1) == len(loader):
            if scaler is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CFG["grad_accum_steps"]
        valid_batches += 1

    return total_loss / max(valid_batches, 1)


## 13. ▶️ Run Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"], eps=1e-6, weight_decay=0.01)

updates_per_epoch = (len(train_loader) + CFG["grad_accum_steps"] - 1) // CFG["grad_accum_steps"]
total_steps = updates_per_epoch * CFG["epochs"]
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler("cuda") if USE_AMP else None

best_f1 = 0.0
checkpoint_path = os.path.join(CFG["output_dir"], "model_weights_v2.pt")
history = {"train_loss": [], "val_f1": []}

print("\n--- Training Started ---")
for epoch in range(CFG["epochs"]):
    print(f"\n[Epoch {epoch + 1}/{CFG['epochs']}]")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device, criterion, scaler)
    val_f1, val_detail = evaluate_macro_f1(model, tokenizer, val_ids, val_df, device, data_dir=TXT_DIR)

    history["train_loss"].append(train_loss)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch + 1} -> Train Loss: {train_loss:.4f} | Val Macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  ⭐ Best model saved: {checkpoint_path}")

print(f"\n✅ Training finished. Best Val Macro-F1: {best_f1:.4f}")



--- Training Started ---

[Epoch 1/3]


Training:   0%|          | 0/2373 [00:00<?, ?it/s]

/tmp/ipykernel_9277/523297737.py:34: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Evaluating Val F1:   0%|          | 0/147 [00:00<?, ?it/s]

Epoch 1 -> Train Loss: 1.0162 | Val Macro-F1: 0.3033
  ⭐ Best model saved: outputs/model_weights_v2.pt

[Epoch 2/3]


Training:   0%|          | 0/2373 [00:00<?, ?it/s]

## 14. 📈 Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], marker='o', color='#0984e3')
ax1.set_title("Training Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(range(1, len(history["val_f1"]) + 1), history["val_f1"], marker='s', color='#00b894')
ax2.set_title("Validation Macro-F1")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Macro-F1")
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "training_curves_v2.png"), dpi=200)
plt.show()


## 15. 🏆 Final Per-Class Evaluation

Loads the best checkpoint and re-runs evaluation for a clean per-class breakdown.

In [ ]:
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"⭐ Loaded best weights: {checkpoint_path}")

final_f1, final_detail = evaluate_macro_f1(model, tokenizer, val_ids, val_df, device, data_dir=TXT_DIR)

print("=" * 70)
print(f"🏆 FINAL VALIDATION MACRO-F1: {final_f1:.4f}")
print("=" * 70)
display(final_detail)

plt.figure(figsize=(10, 4.5))
plt.barh(final_detail["Discourse Type"], final_detail["F1-Score"], color='#6c5ce7')
plt.axvline(final_f1, color='red', linestyle='--', label=f'Macro-F1 ({final_f1:.4f})')
plt.xlabel("F1-Score")
plt.xlim(0, 1.0)
plt.title("Baseline 2 (Sliding Window) — Validation F1 per Class")
plt.legend(loc='lower right')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
